# Visualizing points, lines, dot products, and cross products

This notebook explains the projective-geometry rules used in computer vision:

- `line dot point = 0` means the point is on the line.
- `point cross point = line` gives the line through two points.
- `line cross line = point` gives the intersection point of two lines.

The trick is that we represent ordinary 2D points and lines as 3D homogeneous vectors.

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    from ipywidgets import interact, FloatSlider
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

plt.rcParams['figure.figsize'] = (7, 7)
plt.rcParams['axes.grid'] = True

def H(x, y):
    """Convert a 2D point (x, y) into homogeneous form (x, y, 1)."""
    return np.array([x, y, 1.0], dtype=float)

def normalize_point(p):
    """Convert homogeneous point (x, y, w) back to ordinary 2D when w is not zero."""
    p = np.array(p, dtype=float)
    if abs(p[2]) < 1e-9:
        return p
    return p / p[2]

def normalize_line(l):
    """Scale line coefficients so a and b have length 1. Same line, cleaner numbers."""
    l = np.array(l, dtype=float)
    n = np.linalg.norm(l[:2])
    if n < 1e-9:
        return l
    return l / n

def line_label(l):
    a, b, c = l
    return f"{a:.2f}x + {b:.2f}y + {c:.2f} = 0"

def plot_line(ax, l, xlim=(-1, 6), ylim=(-1, 6), label=None, color='C0', lw=2):
    """Plot homogeneous line l=(a,b,c), meaning ax + by + c = 0."""
    a, b, c = l
    xs = np.linspace(xlim[0], xlim[1], 400)
    if abs(b) > 1e-9:
        ys = -(a * xs + c) / b
        ax.plot(xs, ys, color=color, lw=lw, label=label)
    elif abs(a) > 1e-9:
        x = -c / a
        ax.axvline(x, color=color, lw=lw, label=label)

def setup_axis(ax, xlim=(-1, 6), ylim=(-1, 6), title=None):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect('equal', adjustable='box')
    ax.axhline(0, color='black', lw=1)
    ax.axvline(0, color='black', lw=1)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    if title:
        ax.set_title(title)

def draw_point(ax, p, label, color='C1', s=80):
    p = normalize_point(p)
    ax.scatter([p[0]], [p[1]], color=color, s=s, zorder=5)
    ax.annotate(label, (p[0], p[1]), xytext=(8, 8), textcoords='offset points', fontsize=12)

## 2. Dot product using cosine

For two vectors `a` and `b`:

`a dot b = |a| |b| cos(theta)`

If the vectors point in the same direction, `theta = 0`, so `cos(theta) = 1`.

In [ ]:
a = np.array([2, 2])
b = np.array([1, 1])

dot_direct = np.dot(a, b)
norm_a = np.linalg.norm(a)
norm_b = np.linalg.norm(b)
cos_theta = dot_direct / (norm_a * norm_b)
theta = np.degrees(np.arccos(np.clip(cos_theta, -1, 1)))
dot_cos = norm_a * norm_b * cos_theta

print(f"a = {a}, b = {b}")
print(f"|a| = {norm_a:.4f}")
print(f"|b| = {norm_b:.4f}")
print(f"theta = {theta:.2f} degrees")
print(f"a dot b = |a||b|cos(theta) = {dot_cos:.4f}")

fig, ax = plt.subplots()
setup_axis(ax, xlim=(-0.5, 3), ylim=(-0.5, 3), title='Dot product: same direction means positive and large')
ax.quiver(0, 0, a[0], a[1], angles='xy', scale_units='xy', scale=1, color='C0', label='a = (2,2)')
ax.quiver(0, 0, b[0], b[1], angles='xy', scale_units='xy', scale=1, color='C1', label='b = (1,1)')
ax.legend()
plt.show()

## 3. A line is a vector too

A 2D point `(x, y)` becomes a homogeneous point:

`p = (x, y, 1)`

A 2D line becomes a homogeneous vector:

`l = (a, b, c)`

That vector represents the graph equation:

`ax + by + c = 0`

## 4. Why line dot point equals zero

If `l = (a, b, c)` and `p = (x, y, 1)`, then:

`l dot p = ax + by + c`

So `l dot p = 0` means exactly: the point satisfies the line equation.

In [ ]:
l = np.array([1, -1, 0])   # x - y = 0, so y = x
p_on = H(2, 2)
p_off = H(2, 3)

print('Line l =', l, 'means x - y = 0, or y = x')
print('Point on line:', p_on, ' l dot p =', np.dot(l, p_on))
print('Point off line:', p_off, ' l dot p =', np.dot(l, p_off))

fig, ax = plt.subplots()
setup_axis(ax, title='A point is on the line exactly when l dot p = 0')
plot_line(ax, l, label='l: x - y = 0', color='C0')
draw_point(ax, p_on, 'p=(2,2), dot=0', color='C2')
draw_point(ax, p_off, 'q=(2,3), dot=-1', color='C3')
ax.legend()
plt.show()

## 5. Why point cross point gives a line

The cross product `p1 cross p2` creates a vector perpendicular to both input point-vectors.

Call that result `l`:

`l = p1 cross p2`

Because `l` is perpendicular to both points:

`l dot p1 = 0` and `l dot p2 = 0`

But `line dot point = 0` means the point is on the line. Therefore, `l` is the line passing through both points.

In [ ]:
p1 = H(1, 1)
p2 = H(4, 3)
l_from_points = np.cross(p1, p2)

print('p1 =', p1)
print('p2 =', p2)
print('l = p1 cross p2 =', l_from_points)
print('Line equation:', line_label(l_from_points))
print('l dot p1 =', np.dot(l_from_points, p1))
print('l dot p2 =', np.dot(l_from_points, p2))

fig, ax = plt.subplots()
setup_axis(ax, title='point cross point gives the line through both points')
plot_line(ax, l_from_points, label='p1 cross p2', color='C0')
draw_point(ax, p1, 'p1=(1,1)', color='C1')
draw_point(ax, p2, 'p2=(4,3)', color='C2')
ax.legend()
plt.show()

## 6. Why line cross line gives an intersection point

Now reverse the idea. If we have two lines:

`l1 = (a1, b1, c1)`

`l2 = (a2, b2, c2)`

Their cross product creates a point-vector perpendicular to both line-vectors:

`p = l1 cross l2`

So:

`l1 dot p = 0` and `l2 dot p = 0`

That means `p` lies on both lines. A point that lies on both lines is their intersection.

In [ ]:
l1 = np.array([1, -1, 0])    # x - y = 0, y = x
l2 = np.array([1, 1, -4])    # x + y - 4 = 0
p_intersection = np.cross(l1, l2)
p_intersection_normalized = normalize_point(p_intersection)

print('l1 =', l1, 'means', line_label(l1))
print('l2 =', l2, 'means', line_label(l2))
print('p = l1 cross l2 =', p_intersection)
print('normalized p =', p_intersection_normalized)
print('l1 dot p =', np.dot(l1, p_intersection))
print('l2 dot p =', np.dot(l2, p_intersection))

fig, ax = plt.subplots()
setup_axis(ax, title='line cross line gives the intersection point')
plot_line(ax, l1, label='l1: y = x', color='C0')
plot_line(ax, l2, label='l2: x + y = 4', color='C1')
draw_point(ax, p_intersection, 'intersection (2,2)', color='C3')
ax.legend()
plt.show()

## 7. Interactive: move two points and watch the line change

Change the point coordinates. The notebook recomputes:

`l = p1 cross p2`

Then it checks that both dot products are zero.

In [ ]:
def demo_point_cross_point(x1=1.0, y1=1.0, x2=4.0, y2=3.0):
    p1 = H(x1, y1)
    p2 = H(x2, y2)
    l = np.cross(p1, p2)

    fig, ax = plt.subplots()
    setup_axis(ax, title='Move the points: p1 cross p2 is the line through them')
    if np.linalg.norm(l[:2]) > 1e-9:
        plot_line(ax, l, label='line = p1 cross p2', color='C0')
    draw_point(ax, p1, 'p1', color='C1')
    draw_point(ax, p2, 'p2', color='C2')
    ax.legend()
    plt.show()

    print('p1 =', p1)
    print('p2 =', p2)
    print('line l = p1 cross p2 =', l)
    print('line equation:', line_label(l))
    print('l dot p1 =', np.dot(l, p1))
    print('l dot p2 =', np.dot(l, p2))

if WIDGETS_AVAILABLE:
    interact(
        demo_point_cross_point,
        x1=FloatSlider(value=1, min=-1, max=6, step=0.5),
        y1=FloatSlider(value=1, min=-1, max=6, step=0.5),
        x2=FloatSlider(value=4, min=-1, max=6, step=0.5),
        y2=FloatSlider(value=3, min=-1, max=6, step=0.5),
    )
else:
    demo_point_cross_point()

## 8. Interactive: move two lines and watch the intersection change

Each line has the form:

`ax + by + c = 0`

The notebook recomputes:

`p = l1 cross l2`

Then it checks that `p` lies on both lines.

In [ ]:
def demo_line_cross_line(a1=1.0, b1=-1.0, c1=0.0, a2=1.0, b2=1.0, c2=-4.0):
    l1 = np.array([a1, b1, c1])
    l2 = np.array([a2, b2, c2])
    p = np.cross(l1, l2)
    pn = normalize_point(p)

    fig, ax = plt.subplots()
    setup_axis(ax, title='Move the lines: l1 cross l2 is their intersection')
    plot_line(ax, l1, label='l1', color='C0')
    plot_line(ax, l2, label='l2', color='C1')
    if abs(p[2]) > 1e-9:
        draw_point(ax, p, f'intersection ({pn[0]:.2f},{pn[1]:.2f})', color='C3')
    else:
        ax.text(0.05, 0.95, 'Parallel lines: intersection is at infinity', transform=ax.transAxes, va='top')
    ax.legend()
    plt.show()

    print('l1 =', l1, '=>', line_label(l1))
    print('l2 =', l2, '=>', line_label(l2))
    print('p = l1 cross l2 =', p)
    if abs(p[2]) > 1e-9:
        print('normalized point =', pn)
    else:
        print('w = 0, so the intersection is a point at infinity. This is how projective geometry handles parallel lines.')
    print('l1 dot p =', np.dot(l1, p))
    print('l2 dot p =', np.dot(l2, p))

if WIDGETS_AVAILABLE:
    interact(
        demo_line_cross_line,
        a1=FloatSlider(value=1, min=-3, max=3, step=0.5),
        b1=FloatSlider(value=-1, min=-3, max=3, step=0.5),
        c1=FloatSlider(value=0, min=-6, max=6, step=0.5),
        a2=FloatSlider(value=1, min=-3, max=3, step=0.5),
        b2=FloatSlider(value=1, min=-3, max=3, step=0.5),
        c2=FloatSlider(value=-4, min=-6, max=6, step=0.5),
    )
else:
    demo_line_cross_line()

## 9. Big picture

The whole story is one repeated idea:

`line dot point = 0`

means the point lies on the line.

The cross product gives a vector perpendicular to two input vectors. Therefore:

- `point cross point` creates a line vector perpendicular to both points, so both points lie on that line.
- `line cross line` creates a point vector perpendicular to both lines, so that point lies on both lines.

That is why the same operation can create a line in one case and an intersection point in the other.